Importar os dados

In [114]:
#importar dados brutos

import pandas as pd

data2023a1 = pd.read_csv(
    "dados_brutos/area1_2023.csv",
    encoding="latin-1",
    sep=";",
)

data2023a2 = pd.read_csv(
    "dados_brutos/area2_2023.csv",
    encoding="latin-1",
    sep=";",
)

data2024a1 = pd.read_csv(
    "dados_brutos/area1_2024.csv",
    encoding="latin-1",
    sep=";",
)

data2024a2 = pd.read_csv(
    "dados_brutos/area2_2024.csv",
    encoding="latin-1",
    sep=";",
)

data2023temp = pd.read_csv(
    "dados_brutos/INMET_SE_SP_A711_SAO CARLOS_01-01-2023_A_31-12-2023.CSV",
    encoding="latin-1",
    sep=";",
    skiprows=8
)

data2024temp = pd.read_csv(
    "dados_brutos/INMET_SE_SP_A711_SAO CARLOS_01-01-2024_A_31-12-2024.CSV",
    encoding="latin-1",
    sep=";",
    skiprows=8
)

data2025temp = pd.read_csv(
    "dados_brutos/INMET_SE_SP_A711_SAO CARLOS_01-01-2025_A_31-12-2025.CSV",
    encoding="latin-1",
    sep=";",
    skiprows=8
)

Tratamento dos arquivos de demanda

In [115]:
#Padronizar a grafia dos dias da semana

data2023a1['Dia_semana'] = data2023a1['Dia_semana'].str.lower()
data2024a1['Dia_semana'] = data2024a1['Dia_semana'].str.lower()
#data2025a1['Dia_semana'] = data2025a1['Dia_semana'].str.lower()
data2023a2['Dia_semana'] = data2023a2['Dia_semana'].str.lower()
data2024a2['Dia_semana'] = data2024a2['Dia_semana'].str.lower()
#data2025a2['Dia_semana'] = data2025a2['Dia_semana'].str.lower()

In [116]:
def transformar(df, ano, area):

    if area == 1:
        df = df[['Mês', 'Dia_semana', 'Dia_mês', 'Total_almoço', 'Total_jantar']]

        df_long = df.melt(
            id_vars=['Mês', 'Dia_semana', 'Dia_mês'],
            value_vars=['Total_almoço', 'Total_jantar'],
            var_name='refeicao',
            value_name='total'
        )

        df_long['refeicao'] = df_long['refeicao'].replace({
            'Total_almoço': 'almoco',
            'Total_jantar': 'jantar'
        })

        ordem = ['almoco', 'jantar']
        df_long['refeicao'] = pd.Categorical(
            df_long['refeicao'],
            categories=ordem,
            ordered=True
        )

    else:  # área 2

        df = df[['Mês', 'Dia_semana', 'Dia_mês', 'Total_almoço']]
        df_long = df.rename(columns={'Total_almoço': 'total'})
        df_long['refeicao'] = 'almoco'

    # adiciona coluna ano
    df_long.insert(0, 'Ano', ano)

    # ordenação
    df_long = df_long.sort_values(
        by=['Ano', 'Mês', 'Dia_mês', 'refeicao', 'total']
    ).reset_index(drop=True)

    df_long = df_long[['Ano', 'Mês', 'Dia_semana', 'Dia_mês', 'refeicao', 'total']]
    return df_long

In [117]:
data2023a1 = transformar(data2023a1,2023,1)
data2024a1 = transformar(data2024a1,2024,1)
#data2025a1 = transformar(data2025a1,2025,1)

data2023a2 = transformar(data2023a2,2023,2)
data2024a2= transformar(data2024a2,2024,2)
#data2025a2= transformar(data2025a2,2025,2)

In [119]:
data2023a1.head(10)

,Ano,Mês,Dia_semana,Dia_mês,refeicao,total
0,2023,1,dom,1,almoco,0.0
1,2023,1,dom,1,jantar,0.0
2,2023,1,seg,2,almoco,0.0
3,2023,1,seg,2,jantar,0.0
4,2023,1,ter,3,almoco,207.0
5,2023,1,ter,3,jantar,183.0
6,2023,1,qua,4,almoco,247.0
7,2023,1,qua,4,jantar,181.0
8,2023,1,qui,5,almoco,281.0
9,2023,1,qui,5,jantar,248.0


Tratamento dos arquivos de informações meteorológicas

In [31]:
def processa_temp(df):

    # Remove colunas que não serão usadas
    df = df.drop(columns=[
        'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
        'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
        'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
        'RADIACAO GLOBAL (Kj/m²)',
        'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
        'TEMPERATURA DO PONTO DE ORVALHO (°C)',
        'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
        'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
        'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
        'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
        'Unnamed: 19'
    ], errors='ignore')

    # Cria datetime
    df['datetime'] = pd.to_datetime(
        df['Data'] + ' ' + df['Hora UTC'],
        format='%Y/%m/%d %H%M UTC',
        errors='coerce'
    )

    # Extrai data e hora
    df['data'] = df['datetime'].dt.date
    df['hora'] = df['datetime'].dt.hour

    # Colunas numéricas
    colunas_numericas = [
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
        'UMIDADE RELATIVA DO AR, HORARIA (%)',
        'VENTO, VELOCIDADE HORARIA (m/s)',
        'VENTO, RAJADA MAXIMA (m/s)'
    ]

    # Converte para numérico
    for col in colunas_numericas:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.', regex=False),
            errors='coerce'
        )

    # Renomeia colunas
    df = df.rename(columns={
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'Precipitacao_mm',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'Temp_max_C',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'Temp_min_C',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'Umid_rel_ar',
        'VENTO, RAJADA MAXIMA (m/s)': 'Vento_rajada_maxima_ms',
        'VENTO, VELOCIDADE HORARIA (m/s)': 'Vento_velocidade_ms'
    })

    # Atualiza lista de colunas
    colunas_numericas = [
        'Precipitacao_mm',
        'Temp_max_C',
        'Temp_min_C',
        'Umid_rel_ar',
        'Vento_velocidade_ms',
        'Vento_rajada_maxima_ms'
    ]

    # Classifica refeição
    def classifica_refeicao(h):
        if 10 <= h <= 11:
            return 'cafe da manha'
        elif 14 <= h <= 16:
            return 'almoco'
        elif 20 <= h <= 22:
            return 'jantar'
        else:
            return None

    df['refeicao'] = df['hora'].apply(classifica_refeicao)

    # Mantém só refeições desejadas
    df = df[df['refeicao'].notna()]

    # Agrupa
    df_refeicao = (
        df.groupby(['data', 'refeicao'])[colunas_numericas]
        .mean()
        .reset_index()
    )

    # Data → Ano/Mês/Dia
    df_refeicao['data'] = pd.to_datetime(df_refeicao['data'])
    df_refeicao['Ano'] = df_refeicao['data'].dt.year
    df_refeicao['Mês'] = df_refeicao['data'].dt.month
    df_refeicao['Dia_mês'] = df_refeicao['data'].dt.day

    # Reorganiza colunas
    df_refeicao = df_refeicao[['Ano','Mês','Dia_mês','refeicao'] + colunas_numericas]

    ordem_refeicao = ['cafe da manha','almoco','jantar']
    df_refeicao['refeicao'] = pd.Categorical(df_refeicao['refeicao'], categories=ordem_refeicao, ordered=True)

    df_refeicao = df_refeicao.sort_values(by=['Ano','Mês','Dia_mês','refeicao']).reset_index(drop=True)

    return df_refeicao

In [32]:
#Aplica a função

data2023_meteorologia = processa_temp(data2023temp)
data2024_meteorologia = processa_temp(data2024temp)
data2025_meteorologia = processa_temp(data2025temp)

In [39]:
data2023_meteorologia.head()

,Ano,Mês,Dia_mês,refeicao,Precipitacao_mm,Temp_max_C,Temp_min_C,Umid_rel_ar,Vento_velocidade_ms,Vento_rajada_maxima_ms
0,2023,1,1,cafe da manha,0.0,21.900000,20.400000,83.000000,1.200000,3.650000
1,2023,1,1,almoco,0.0,28.333333,25.766667,58.666667,1.000000,4.633333
2,2023,1,1,jantar,0.0,26.333333,24.966667,60.000000,0.833333,3.800000
3,2023,1,2,cafe da manha,0.0,21.750000,19.900000,84.500000,2.600000,6.500000
4,2023,1,2,almoco,0.0,27.900000,25.633333,65.000000,2.000000,6.666667
